In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import spacy
import optuna
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("LogisticRegression HP Tuning with Custom Features")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/84a8f7afc8434d5398ba8feeab10e4f4', creation_time=1784660079577, effective_trace_archival_retention=None, experiment_id='14', last_update_time=1784660079577, lifecycle_stage='active', name='LogisticRegression HP Tuning with Custom Features', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [2]:
# Load dataset
dataset = pd.read_csv('dataset.csv')

# Drop rows with NaN values in 'clean_comment'
cleaned_dataset = dataset.dropna()

In [3]:
# Separate features and target
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Split the cleaned data into train and test sets (80-20 split)
X_train_cleaned, X_test_cleaned, y_train_cleaned, y_test_cleaned = train_test_split(X_cleaned, y_cleaned, test_size=0.2, random_state=42)


In [4]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [5]:
# All 17 universal POS tags, in a FIXED order.
# Looping over this (instead of set(pos_tags)) guarantees every comment produces the
# same 17 POS columns in the same order -> train, test, and future data always line up.
UNIVERSAL_POS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM',
                 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc]

    # 1. Comment Length (number of characters)
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. Unique Word Count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags - loop over the FIXED list (not set()) so every comment
    #    yields all 17 columns in the same order; tags not present come out as 0.
    pos_tags = [token.pos_ for token in doc]
    if word_count > 0:
        pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in UNIVERSAL_POS}
    else:
        pos_proportion = {tag: 0 for tag in UNIVERSAL_POS}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion  # Flattening the POS proportions
    }

In [6]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train_cleaned])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test_cleaned])

In [7]:
train_custom_features.head()

,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_count,ADJ,ADP,ADV,AUX,...,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X
0,51,7,6.428571,7,1.000000,7,0.000000,0.142857,0.142857,0.142857,...,0.142857,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.428571,0.0
1,36,6,5.166667,6,1.000000,6,0.000000,0.000000,0.000000,0.000000,...,0.333333,0.000000,0.000000,0.166667,0.166667,0.0,0.0,0.0,0.333333,0.0
2,64,9,6.222222,9,1.000000,9,0.222222,0.000000,0.111111,0.000000,...,0.444444,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.222222,0.0
3,108,15,6.266667,14,0.933333,15,0.200000,0.000000,0.000000,0.000000,...,0.466667,0.066667,0.066667,0.000000,0.000000,0.0,0.0,0.0,0.200000,0.0
4,6,1,6.000000,1,1.000000,1,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0


In [8]:
# Replace NaN values in POS tag proportions with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [9]:
test_custom_features.isnull().sum()

comment_length       0
word_count           0
avg_word_length      0
unique_word_count    0
lexical_diversity    0
pos_count            0
ADJ                  0
ADP                  0
ADV                  0
AUX                  0
CCONJ                0
DET                  0
INTJ                 0
NOUN                 0
NUM                  0
PART                 0
PRON                 0
PROPN                0
PUNCT                0
SCONJ                0
SYM                  0
VERB                 0
X                    0
dtype: int64

In [10]:
# Apply TfidfVectorizer with bigram setting and max_features=10000
ngram_range = (1, 2)
max_features = 10000
tfidf = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_tfidf = tfidf.fit_transform(X_train_cleaned)
X_test_tfidf = tfidf.transform(X_test_cleaned)

In [11]:
# Keep TF-IDF SPARSE - no .toarray() here.
# The TF-IDF matrix is ~99.8% zeros; densifying it would cost ~2.35 GB and force the
# solver to multiply by zero hundreds of millions of times per iteration.
#
# All we need before stacking is that the small 23-column custom-feature block has the
# same columns in the same order for train and test.
test_custom_features = test_custom_features.reindex(
    columns=train_custom_features.columns, fill_value=0
)

print('custom feature columns aligned:', list(train_custom_features.columns) == list(test_custom_features.columns))

custom feature columns aligned: True


In [12]:
from scipy.sparse import hstack, csr_matrix

# Combine [sparse TF-IDF | small dense custom-feature block] WITHOUT densifying anything.
# Column order is guaranteed: TF-IDF columns come from the fitted vocabulary (identical for
# train/test), and the custom block was aligned in the previous cell. Sparse matrices carry
# no column names, so the old "feature names must be in the same order" error cannot occur.
X_train_combined = hstack([X_train_tfidf, csr_matrix(train_custom_features.values)]).tocsr()
X_test_combined  = hstack([X_test_tfidf,  csr_matrix(test_custom_features.values)]).tocsr()

dense_mb = X_train_combined.shape[0] * X_train_combined.shape[1] * 8 / 1e6
sparse_mb = (X_train_combined.data.nbytes + X_train_combined.indices.nbytes + X_train_combined.indptr.nbytes) / 1e6
print('train shape :', X_train_combined.shape)
print('test  shape :', X_test_combined.shape)
print(f'sparse size : {sparse_mb:.1f} MB   (dense would be ~{dense_mb:.0f} MB)')
print(f'density     : {X_train_combined.nnz / (X_train_combined.shape[0] * X_train_combined.shape[1]) * 100:.2f}%  non-zero')

train shape : (29329, 10023)
test  shape : (7333, 10023)
sparse size : 9.7 MB   (dense would be ~2352 MB)
density     : 0.27%  non-zero


In [13]:
X_train_combined

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 796623 stored elements and shape (29329, 10023)>

In [14]:
X_train = X_train_combined
X_test = X_test_combined
y_train = y_train_cleaned
y_test = y_test_cleaned

In [15]:
# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number, log_model=False):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("scaler", "MaxAbsScaler")

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Only log the model artifact for the best run to stay within DagShub free tier limits
        if log_model:
            # Persist locally first so a long train is never lost to a logging failure
            joblib.dump(model, f"{model_name}_best_model.joblib")
            try:
                mlflow.sklearn.log_model(
                    model, f"{model_name}_model",
                    serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
                )
            except Exception as e:
                print(f"[warn] model artifact logging failed (metrics/params + joblib file are safe): {e}")

        return accuracy

In [16]:
# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    # Hyperparameter space to explore
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)  # inverse regularization strength
    # sklearn >= 1.8 deprecated `penalty` in favour of a continuous `l1_ratio` dial:
    #   0.0 = pure L2   |   0.5 = elasticnet blend   |   1.0 = pure L1
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
    # Class imbalance handling - let Optuna decide whether balancing helps
    class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])

    # Log trial parameters
    params = {
        'C': C,
        'l1_ratio': l1_ratio,
        'class_weight': class_weight,
        'solver': 'saga',   # saga supports the full l1/l2/elasticnet range
        'max_iter': 300
    }

    # MaxAbsScaler -> LogisticRegression pipeline.
    # MaxAbsScaler is sparse-safe (unlike StandardScaler, which would destroy sparsity).
    # No `penalty` argument - l1_ratio fully controls the regularization type.
    model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=C, l1_ratio=l1_ratio, class_weight=class_weight,
                                   solver='saga', max_iter=300, random_state=42))
    ])

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LogisticRegression", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy

In [17]:
# Step 7: Run Optuna for Logistic Regression, log the best model, and plot the importance of each parameter
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=50)  # 50 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = Pipeline([
        ('scaler', MaxAbsScaler()),
        ('clf', LogisticRegression(C=best_params['C'],
                                   l1_ratio=best_params['l1_ratio'],
                                   class_weight=best_params['class_weight'],
                                   solver='saga', max_iter=300, random_state=42))
    ])

    # Log the best model with MLflow (only this run uploads the model artifact)
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test, best_params, "Best", log_model=True)

    # Plots are optional - never let a missing optional dependency kill the return value
    try:
        optuna.visualization.plot_param_importances(study).show()
        optuna.visualization.plot_optimization_history(study).show()
    except Exception as e:
        print(f"[warn] optuna plots skipped: {e}")

    print(f"\nBest accuracy: {study.best_value:.4f}")
    print(f"Best params  : {best_params}")

    # Return the best model so it can be inspected outside this function
    return best_model

In [18]:
# Run the experiment for Logistic Regression
best_model = run_optuna_experiment()

[I 2026-07-22 03:06:50,136] A new study created in memory with name: no-name-9817b509-d3fd-4e21-89d3-de95b1bf5e78
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run Trial_0_LogisticRegression_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14/runs/e6326f0f8b4449a8b27d7d5971da678a
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14


[I 2026-07-22 03:23:55,275] Trial 0 finished with value: 0.8703122869221328 and parameters: {'C': 2.2833658272215223, 'l1_ratio': 0.8479244400135727, 'class_weight': None}. Best is trial 0 with value: 0.8703122869221328.
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run Trial_1_LogisticRegression_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14/runs/d4db8aed67b84687ad35328e03a795f1
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14


[I 2026-07-22 03:48:36,866] Trial 1 finished with value: 0.8446747579435429 and parameters: {'C': 0.6828195620906322, 'l1_ratio': 0.2724035335474214, 'class_weight': None}. Best is trial 0 with value: 0.8703122869221328.


...

🏃 View run Trial_48_LogisticRegression_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14/runs/883fe42fd29746e98f35dfb3a3ab9df5
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14


[I 2026-07-22 10:55:09,177] Trial 48 finished with value: 0.8850402291013227 and parameters: {'C': 1.470580864473077, 'l1_ratio': 0.996647029149772, 'class_weight': None}. Best is trial 22 with value: 0.8870857766262102.
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


🏃 View run Trial_49_LogisticRegression_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14/runs/8f722d27b3164ffca48d811c5bffa0cb
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14


[I 2026-07-22 11:11:15,284] Trial 49 finished with value: 0.8634937951725078 and parameters: {'C': 3.807554386583161, 'l1_ratio': 0.9050200434277258, 'class_weight': None}. Best is trial 22 with value: 0.8870857766262102.
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2026/07/22 11:16:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/22 11:16:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Trial_Best_LogisticRegression_ at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14/runs/6757da8d437f4f29b9774f97bb1fbec5
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/14
[warn] optuna plots skipped: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Best accuracy: 0.8871
Best params  : {'C': 1.1199134843600127, 'l1_ratio': 0.9990014673384794, 'class_weight': None}


In [19]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[-1, 0, 1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10023
,"copy copy: bool, default=TrueSet to False to perform inplace scaling and avoid a copy (if the inputis already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data to [-1, 1].Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the [-1, 1] range, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data.",False
Name,Type,Value
"max_abs_ max_abs_: ndarray of shape (n_features,)Per feature maximum absolute value.","ndarray[float64](10023,)","[1. ,0.55,0.42,...,0.01,1. ,1. ]"


# 📌 Best Params — LogisticRegression + Custom Features

**Run:** 50/50 Optuna trials completed · ~500 minutes (~10 hours)
**Experiment (MLflow):** `LogisticRegression HP Tuning with Custom Features`
**Features:** TF-IDF bigram (1,2), `max_features=10000` + spaCy custom features (23 cols) → 10,023 total, kept **sparse** (9.7 MB vs ~2352 MB dense, 0.27% non-zero)
**Split:** 80/20, `random_state=42`, **no stratify** → 29,329 train / 7,333 test
**Scaler:** MaxAbsScaler (sparse-safe)

## 🏆 Best result

**Test accuracy: `0.8871`**

```python
best_params_logreg_custom = {
    'C':            1.1199134843600127,
    'l1_ratio':     0.9990014673384794,   # ~1.0 => essentially pure L1
    'class_weight': None,
    # fixed (not tuned)
    'solver':       'saga',
    'max_iter':     300,
    'random_state': 42,
}
```

Rebuild with:
```python
Pipeline([
    ('scaler', MaxAbsScaler()),
    ('clf', LogisticRegression(C=1.1199134843600127,
                               l1_ratio=0.9990014673384794,
                               class_weight=None,
                               solver='saga', max_iter=300, random_state=42))
])
```

## Full metrics

| Metric | Value |
|---|---|
| Accuracy | 0.8871 |
| Macro Precision | 0.8833 |
| Macro Recall | 0.8726 |
| **Macro F1** | **0.8756** |
| Weighted Precision | 0.8877 |
| Weighted Recall | 0.8871 |
| Weighted F1 | 0.8855 |
| -1 Precision / Recall / F1 | 0.8688 / 0.7596 / 0.8105 |
| 0 Precision / Recall / F1 | 0.8680 / 0.9721 / 0.9171 |
| 1 Precision / Recall / F1 | 0.9130 / 0.8860 / 0.8993 |

## Observations

- **`l1_ratio ≈ 0.999` means the winner is essentially pure L1 regularization.** This matters: earlier notebooks passed `penalty='l1'`, but on sklearn ≥1.8 `penalty` is deprecated and `l1_ratio` (default `0.0` = pure L2) silently takes precedence. So those runs actually trained **L2** despite asking for L1. This is the first run to genuinely use L1 — and L1 clearly won.
- **`class_weight=None` was selected** because the Optuna objective was **accuracy**. On imbalanced data, quietly under-serving the minority class is the cheapest way to raise accuracy. Visible in the results: strong class 0/1 scores, weakest recall on class -1 (0.7596).
- Tuning mattered far more than expected: trial 0 scored 0.8703, best reached **0.8871** (+1.68 accuracy, +1.78 macro F1).
- Keeping the matrix **sparse** was essential — the dense version could not finish a single trial in over an hour.

## ⚠️ Caveats before comparing with other experiments

1. **Different test rows.** This split has **no `stratify`**; `experiment_5_lor_with_hpt` uses `stratify=df['category']`. Same size (7,333) but *different comments* → ±0.5–1 point of split luck.
2. **Different regularization in practice.** exp_5 effectively trained L2 (see note above); this run trained true L1.
3. **Unequal trial budgets.** 50 trials here vs 30 in exp_5 — more attempts at the test set means more optimistic bias.
4. **Selected on the test set.** Optuna maximised accuracy *on this test set*, so `0.8871` is optimistic, not a clean generalisation estimate.

→ A fair verdict needs **both models trained on one shared split and scored on one shared held-out evaluation set.**
